# Notebook 2: Search Engine Development & Matching EvaluationThis notebook implements the vector space search engine (TF-IDF & Cosine Similarity)and evaluates query resolution performance across Arabic and English inputs.---

## 1. Environment & Knowledge Base Loading

In [1]:
import osimport jsonimport reimport numpy as npfrom sklearn.feature_extraction.text import TfidfVectorizerfrom sklearn.metrics.pairwise import cosine_similarityKB_PATH = os.path.join('..', 'data', 'insurance_knowledge_base.json')if not os.path.exists(KB_PATH):    KB_PATH = 'insurance_knowledge_base.json'with open(KB_PATH, 'r', encoding='utf-8') as f:    kb = json.load(f)print(f'Loaded knowledge base with {len(kb)} insurance entities.')

Loaded knowledge base with 78 insurance entities.


## 2. Arabic Text Normalization

In [1]:
def normalize_arabic(text):    if not text: return ''    text = re.sub(r'[\u064B-\u0652\u0670]', '', text)    text = re.sub(r'[أإآ]', 'ا', text)    text = text.replace('ة', 'ه').replace('ى', 'ي').lower()    return textsample_words = ['يونايتد', 'المحظورات', 'أقصى مدة', 'صلاحية']for w in sample_words:    print(f'  {w:<15} → {normalize_arabic(w)}')

  يونايتد         → يونايتد
  المحظورات       → المحظورات
  أقصى مدة        → اقصي مده
  صلاحية          → صلاحيه


## 3. TF-IDF Indexing

In [1]:
documents, chunk_index = [], []STOP_WORDS = {'في','من','على','إلى','عن','مع','هذا','هذه','التي','الذي','هو','هي','أن','كان','كل','لم','لن','يتم','يجب','لابد','و','أو','لا','ما','فى','بعد','قبل'}for ck, cd in kb.items():    for cat, pol in cd.get('policies', {}).items():        chunk = ' '.join([cd.get('company_name', ''), cat, pol.get('details', ''), pol.get('notes', '')])        documents.append(normalize_arabic(chunk))        chunk_index.append((ck, cat))vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words=list(STOP_WORDS))tfidf_matrix = vectorizer.fit_transform(documents)print(f'Indexed documents: {tfidf_matrix.shape[0]}')print(f'Vocabulary size: {tfidf_matrix.shape[1]} features')

Indexed documents: 772
Vocabulary size: 5000 features


## 4. Query Resolution Test

In [1]:
def search_policy(query, top_k=3):    q_norm = normalize_arabic(query)    q_vec = vectorizer.transform([q_norm])    sims = cosine_similarity(q_vec, tfidf_matrix).flatten()    top_idx = sims.argsort()[-top_k:][::-1]    results = []    for idx in top_idx:        if sims[idx] < 0.05: continue        ck, cat = chunk_index[idx]        comp = kb[ck]        pol = comp['policies'][cat]        results.append({            'company': comp.get('company_name', ''),            'category': cat,            'score': float(sims[idx]),            'preview': pol.get('details', '')[:100]        })    return resultsqueries = ['محظورات يونايتد', 'أقصى مدة صرف ويبكو', 'تواصل موافقات دريم مشرق', 'copay ALICO']for q in queries:    print(f'\n🔍 Query: "{q}"')    res = search_policy(q, top_k=1)    if res:        print(f"   Matched: {res[0]['company']} | Category: {res[0]['category']} | Score: {res[0]['score']:.3f}")    else:        print('   No match found')


🔍 Query: "محظورات يونايتد"
   Matched: يونايتد-united | Category: التحمل | Score: 0.307

🔍 Query: "أقصى مدة صرف ويبكو"
   Matched: ALICO | Category: أقصى مدة للصرف | Score: 0.302

🔍 Query: "تواصل موافقات دريم مشرق"
   Matched: شركة دريم مشرق للأغذية-dream mashreq | Category: الحد الأقصى | Score: 0.370

🔍 Query: "copay ALICO"
   Matched: ALICO | Category: أقصى مدة للصرف | Score: 0.454
